In [55]:
# %%
# ============================================================
# GPT-5-mini Primary Topic Classification Validation
#
# English AV News Dataset
#
# Task:
# Predict dominant topic (topic_primary)
#
# ============================================================


# %%
# If needed:
# pip install openai python-dotenv


import os
import time
import json
import pandas as pd
import numpy as np


from pathlib import Path

from dotenv import load_dotenv

from openai import OpenAI

In [56]:
# %%
# ============================================================
# Load OpenAI API
# ============================================================


load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)


key = os.getenv("OPENAI_API_KEY")


if key:
    print("API key loaded successfully")
else:
    print("API key NOT found")


client = OpenAI()


MODEL = "gpt-5-mini"

API key loaded successfully


In [57]:
# %%
# ============================================================
# Test API connection
# ============================================================


test_text = """
Waymo expanded its autonomous ride-hailing service
after successful testing in additional areas.
"""


response = client.responses.create(

    model=MODEL,

    input=f"""

Classify the dominant topic of this autonomous vehicle news text.

Return only one topic name.

Text:

{test_text}

"""

)


print(response.output_text)

Service expansion


In [84]:
# %%
# ============================================================
# Load English validation data
# ============================================================


BASE_DIR = Path(
    "/Users/yurujia/Desktop/Dissertation Data/USA"
)


USA_PATH = (
    BASE_DIR /
    "descriptive_stats_USA_Today/"
    "USA_Today_topic_multilabel_annotated_av_relevant_paragraph.xlsx"
)


WSJ_PATH = (
    BASE_DIR /
    "descriptive_stats_WSJ/"
    "WSJ_topic_multilabel_annotated.xlsx"
)



usa = pd.read_excel(
    USA_PATH
)


wsj = pd.read_excel(
    WSJ_PATH
)



print(
    usa.shape,
    wsj.shape
)

(29, 121) (37, 89)


In [59]:
# %%
# ============================================================
# Standardise validation dataset
# ============================================================


usa_validation = pd.DataFrame({

    "source":
        "USA Today",

    "text":
        usa["av_relevant_paragraph"],

    "human_topic":
        usa["topic_primary"]

})


wsj_validation = pd.DataFrame({

    "source":
        "WSJ",

    "text":
        wsj["abstract"],

    "human_topic":
        wsj["topic_primary"]

})



validation = pd.concat(
    [
        usa_validation,
        wsj_validation
    ],
    ignore_index=True
)



print(
    validation.shape
)


display(
    validation.head()
)

(66, 3)


,source,text,human_topic
0,USA Today,"Not that long ago, checking in on the state of...",Technology and Innovation
1,USA Today,Fiat Chrysler's decision to launch an ultra-po...,Business and Commercialisation
2,USA Today,A self-driving car flipped on its side seems g...,Safety and Risk
3,USA Today,So if you're skeptical or nervous about the co...,Mobility and Social Impact
4,USA Today,General Motors said Tuesday it has finished ma...,Technology and Innovation


In [86]:
# %%
# ============================================================
# Human topic distribution
# ============================================================


display(
    validation["human_topic"]
    .value_counts()
)

human_topic
Business and Commercialisation    22
Technology and Innovation         14
Mobility and Social Impact         9
Safety and Risk                    8
Policy and Regulation              4
Legal and Ethics                   3
Public Acceptance and Trust        3
Other                              2
Unclear                            1
Name: count, dtype: int64

In [61]:
# %%
# ============================================================
# Topic categories
# ============================================================


TOPICS = [

"Technology and Innovation",

"Safety and Risk",

"Policy and Regulation",

"Business and Commercialisation",

"Public Acceptance and Trust",

"Mobility and Social Impact",

"Environment and Sustainability",

"Legal and Ethics",

"Other"

]

In [62]:
# %%
# ============================================================
# Topic categories represented in human validation sample
#
# Purpose:
# Macro metrics should be calculated across categories that
# actually appear in the manually labelled validation sample.
#
# Example:
# If Environment and Sustainability has 0 human-labelled cases,
# it cannot meaningfully contribute to validation Macro F1.
# ============================================================


observed_topics = [

    topic

    for topic in TOPICS

    if topic in validation[
        "human_topic"
    ].dropna().unique()

]


print(
    "Observed topics in human validation sample:"
)

for topic in observed_topics:

    print(
        "-",
        topic
    )


print(
    "\nNumber of observed topic categories:",
    len(observed_topics)
)

Observed topics in human validation sample:
- Technology and Innovation
- Safety and Risk
- Policy and Regulation
- Business and Commercialisation
- Public Acceptance and Trust
- Mobility and Social Impact
- Legal and Ethics
- Other

Number of observed topic categories: 8


In [63]:
# %%
# ============================================================
# P1 Basic zero-shot
# ============================================================


def build_prompt_p1(text):

    return f"""

You are performing topic classification for an academic research project
on autonomous vehicle (AV) news coverage.

Identify the single dominant topic of the following news text.

Choose exactly one category:

Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other


Return only the topic name.


Text:

{text}

"""

In [64]:
def build_prompt_p2(text):

    return f"""

You are an expert media analyst specialising in autonomous vehicle (AV) news coverage.

Your task is to identify the SINGLE dominant topic (primary frame) of the following AV-related news article.

Select exactly ONE topic from the predefined categories below.

Do not create new categories.
Do not use alternative wording.
Return only the category name.


Available topics:


1. Technology and Innovation

Choose this topic when the article mainly focuses on the development, improvement, testing, or technical capabilities of autonomous driving technology.

Includes:
- artificial intelligence systems
- sensors, software, algorithms
- vehicle systems
- engineering research
- technical demonstrations
- improvements in autonomous driving capability
- technology testing and validation

Important:
The presence of a company, product, or AV deployment does NOT automatically make the article Business.

Choose Technology when the central question is:
"How does the technology work or improve?"


--------------------------------------------------


2. Safety and Risk

Choose this topic when the article mainly focuses on risks, failures, accidents, reliability, or safety evaluation of autonomous vehicles.

Includes:
- crashes
- collisions
- investigations
- operational failures
- safety concerns
- reliability problems
- risk reduction measures

Important:
Safety concerns should be classified here even if the article also discusses public reactions.

Do NOT classify as Public Acceptance unless the main focus is people's attitudes or willingness to adopt AVs.


--------------------------------------------------


3. Policy and Regulation

Choose this topic when the article mainly focuses on government actions, rules, or regulatory frameworks.

Includes:
- legislation
- government policies
- regulatory approval
- testing permits
- official standards
- public authorities' decisions

Choose this category when the central issue is:
"How are AVs governed or regulated?"


--------------------------------------------------


4. Business and Commercialisation

Choose this topic when the article mainly focuses on economic, corporate, or market activities related to autonomous vehicles.

Includes:
- company strategy
- investment
- acquisitions
- partnerships
- competition between firms
- financial performance
- business models
- industrial development

Important:
Do NOT classify an article as Business only because:
- a company is mentioned;
- an AV test occurs;
- a service operates;
- a robotaxi is deployed.

Choose Business only when the main focus is:
"How companies, markets, or industries develop AV-related business."


--------------------------------------------------


5. Public Acceptance and Trust

Choose this topic when the article mainly focuses on human attitudes, opinions, trust, concerns, or willingness to adopt autonomous vehicles.

Includes:
- consumer acceptance
- public opinion
- trust in AV technology
- fear or hesitation about adoption
- social perception

Important:
A safety incident does not automatically indicate Public Acceptance.
Classify as Safety when the focus is technical or operational risk.


--------------------------------------------------


6. Mobility and Social Impact

Choose this topic when the article mainly focuses on autonomous vehicles as transportation services or their broader effects on mobility and society.

Includes:
- robotaxi services
- autonomous ride-hailing
- passenger transportation
- accessibility
- changes to transportation systems
- urban mobility
- effects on daily life

Important:
Robotaxi and autonomous transportation services should usually be classified as Mobility and Social Impact.

Only classify as Business when the article focuses mainly on company strategy, investment, or market competition.


--------------------------------------------------


7. Environment and Sustainability

Choose this topic when the article mainly focuses on environmental consequences.

Includes:
- emissions reduction
- energy efficiency
- sustainability
- environmental benefits or concerns


--------------------------------------------------


8. Legal and Ethics

Choose this topic when the article mainly focuses on legal responsibility or ethical questions.

Includes:
- liability
- responsibility after accidents
- legal disputes
- ethical dilemmas
- accountability


--------------------------------------------------


Primary topic selection rules:

1.
Choose the topic that best represents the central framing of the article, not all topics mentioned.

2.
Do not classify based on isolated keywords.

3.
When multiple topics appear, identify which issue receives the greatest emphasis.

4.
Distinguish between:
- technology development → Technology and Innovation
- company strategy → Business and Commercialisation
- passenger/service operation → Mobility and Social Impact
- government permission or rules → Policy and Regulation
- accidents or failures → Safety and Risk

5.
Objective news writing does not automatically mean Business or Technology.
Classify according to the substantive issue being discussed.

6.
If the article only briefly mentions autonomous vehicles without a clear thematic focus, choose Other.


Return only ONE topic name.

Text:

{text}

"""

In [65]:
# %%
# ============================================================
# P3 Rule-guided
# ============================================================


def build_prompt_p3(text):

    return f"""

You are classifying the dominant framing of autonomous vehicle news.

Select exactly ONE primary topic.


Available topics:

Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other


Additional primary-topic decision rules:

1.
Select the topic that represents the central framing of the article rather than any secondary themes.

2.
Do not classify based on company names, keywords, or isolated mentions.

3.
Company involvement alone does not indicate Business and Commercialisation.

4.
Distinguish between:
- technical development, engineering progress, testing capability → Technology and Innovation
- company strategy, investment, competition, market activity → Business and Commercialisation

5.
Robotaxi, autonomous ride-hailing, and passenger AV services should usually be classified as Mobility and Social Impact when the focus is transportation services or changes in mobility.

6.
Classify as Public Acceptance and Trust only when the article focuses on public attitudes, trust, concerns, or willingness to adopt autonomous vehicles.

7.
Mentions of skepticism, uncertainty, or concerns do not automatically indicate Public Acceptance. If the article discusses transportation transformation or future mobility, choose Mobility and Social Impact.

8.
Safety incidents, crashes, failures, or reliability issues should be classified as Safety and Risk even when public reaction is mentioned.

9.
Government approval, permits, legislation, and standards should be classified as Policy and Regulation.

10.
Choose only one category based on the dominant framing.


Text:

{text}

"""

In [66]:
# %%
# ============================================================
# API function
# ============================================================


def classify_topic(prompt):

    for attempt in range(3):

        try:

            response = client.responses.create(

                model=MODEL,

                input=prompt

            )


            result = (
                response.output_text
                .strip()
            )


            return result


        except Exception as e:

            print(
                "API error:",
                e
            )

            time.sleep(2)


    return None

In [67]:
# %%
# ============================================================
# Test
# ============================================================


for idx,row in validation.head(5).iterrows():

    print("="*80)

    print(
        row["source"]
    )

    print(
        "Human:",
        row["human_topic"]
    )


    prediction = classify_topic(
        build_prompt_p2(row["text"])
    )


    print(
        "GPT:",
        prediction
    )

USA Today
Human: Technology and Innovation
GPT: Technology and Innovation
USA Today
Human: Business and Commercialisation
GPT: Business and Commercialisation
USA Today
Human: Safety and Risk
GPT: Public Acceptance and Trust
USA Today
Human: Mobility and Social Impact
GPT: Public Acceptance and Trust
USA Today
Human: Technology and Innovation
GPT: Business and Commercialisation


In [68]:
# %%
# ============================================================
# Run all prompts
# ============================================================


results = validation.copy()


for col in [

"P1_EN",

"P2_EN",

"P3_EN"

]:

    results[col]=None



for idx,row in results.iterrows():

    print(
        f"{idx+1}/{len(results)}"
    )


    text=row["text"]


    results.at[idx,"P1_EN"] = classify_topic(
        build_prompt_p1(text)
    )


    results.at[idx,"P2_EN"] = classify_topic(
        build_prompt_p2(text)
    )


    results.at[idx,"P3_EN"] = classify_topic(
        build_prompt_p3(text)
    )




1/66
2/66
3/66
4/66
5/66
6/66
7/66
8/66
9/66
10/66
11/66
12/66
13/66
14/66
15/66
16/66
17/66
18/66
19/66
20/66
21/66
22/66
23/66
24/66
25/66
26/66
27/66
28/66
29/66
30/66
31/66
32/66
33/66
34/66
35/66
36/66
37/66
38/66
39/66
40/66
41/66
42/66
43/66
44/66
45/66
46/66
47/66
48/66
49/66
50/66
51/66
52/66
53/66
54/66
55/66
56/66
57/66
58/66
59/66
60/66
61/66
62/66
63/66
64/66
65/66
66/66


In [69]:
# %%
# ============================================================
# Evaluation
#
# Main validation metrics:
#
# Accuracy:
# Overall proportion of correct classifications
#
# Macro Precision / Recall / F1:
# Calculated across topic categories that are actually
# represented in the human validation sample.
#
# Cohen's Kappa:
# Agreement between GPT and human coding beyond chance.
# ============================================================


from sklearn.metrics import (

    accuracy_score,

    f1_score,

    precision_score,

    recall_score,

    cohen_kappa_score

)


prompt_cols = [

    "P1_EN",

    "P2_EN",

    "P3_EN"

]


evaluation = []



for p in prompt_cols:


    valid = results[
        results[p].notna()
        &
        results["human_topic"].notna()
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        p
    ]


    evaluation.append({

        "prompt":
            p,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


evaluation_df = pd.DataFrame(
    evaluation
)


# Reorder columns for clearer presentation

evaluation_df = evaluation_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


display(
    evaluation_df.round(4)
)

,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_EN,66,0.6818,0.6002,0.6537,0.5848,0.5850
1,P2_EN,66,0.7879,0.6625,0.6927,0.6494,0.7293
2,P3_EN,66,0.7879,0.6690,0.6981,0.6543,0.7316


In [70]:
# %%
# ============================================================
# Confusion matrix for best prompt
# ============================================================


from sklearn.metrics import confusion_matrix


best_prompt="P3_EN"


cm = confusion_matrix(

    results["human_topic"],

    results[best_prompt],

    labels=TOPICS

)



cm_df=pd.DataFrame(

    cm,

    index=TOPICS,

    columns=TOPICS

)


display(
    cm_df
)

,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,12,1,0,1,0,0,0,0,0
Safety and Risk,0,7,0,1,0,0,0,0,0
Policy and Regulation,1,0,3,0,0,0,0,0,0
Business and Commercialisation,1,0,0,19,0,2,0,0,0
Public Acceptance and Trust,0,0,0,1,1,0,0,0,1
Mobility and Social Impact,0,0,0,0,1,8,0,0,0
Environment and Sustainability,0,0,0,0,0,0,0,0,0
Legal and Ethics,0,0,0,1,0,0,0,2,0
Other,1,0,0,0,1,0,0,0,0


In [71]:
# %%
# ============================================================
# Error cases
# ============================================================


errors = results[
    results["human_topic"]
    !=
    results["P3_EN"]
]


print(
    len(errors)
)


display(
    errors[
        [
            "source",
            "text",
            "human_topic",
            "P3_EN"
        ]
    ]
)

14


,source,text,human_topic,P3_EN
3,USA Today,So if you're skeptical or nervous about the co...,Mobility and Social Impact,Public Acceptance and Trust
5,USA Today,"Benchmark, one of the company's largest invest...",Legal and Ethics,Business and Commercialisation
9,USA Today,That's because self-driving cars are safer and...,Technology and Innovation,Safety and Risk
14,USA Today,Musk has also lashed out against the media in ...,Other,Public Acceptance and Trust
16,USA Today,As self-driving vehicle experiments are launch...,Business and Commercialisation,Technology and Innovation
19,USA Today,"Yes, Musk has repeatedly promised that Teslas ...",Business and Commercialisation,Mobility and Social Impact
21,USA Today,"Granted, it can be tougher to assess how impre...",Unclear,Technology and Innovation
27,USA Today,Stellantis has shelved its first Level 3 advan...,Technology and Innovation,Business and Commercialisation
31,WSJ,Uber Technologies has resumed test using its d...,Safety and Risk,Business and Commercialisation
35,WSJ,Rio Tinto is using new technology including se...,Other,Technology and Innovation


In [72]:
# %%
# ============================================================
# Save
# ============================================================


OUTPUT = (

BASE_DIR /

"english_primary_topic_prompt_validation.xlsx"

)


results.to_excel(

    OUTPUT,

    index=False

)


print(
    OUTPUT
)

/Users/yurujia/Desktop/Dissertation Data/USA/english_primary_topic_prompt_validation.xlsx


In [73]:
# %%
# ============================================================
# P3-R: Refined Rule-guided Primary Topic Prompt
#
# Refinement based on error analysis:
# Business over-classification
# Technology vs Business confusion
# Mobility vs Public Acceptance confusion
# ============================================================


def build_prompt_p3r(text):

    return f"""

You are an expert media analyst specialising in autonomous vehicle (AV) news coverage.

Your task is to identify the SINGLE dominant topic (primary frame) of the following AV-related news article.

Select exactly ONE topic from the predefined categories below.

Do not create new categories.
Do not use alternative wording.
Return only the category name.


Available topics:


Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other



--------------------------------------------------

Topic definitions:


Technology and Innovation

Choose this topic when the article mainly focuses on:

- autonomous driving technology development
- AI systems, sensors, software, algorithms
- engineering progress
- technical capabilities
- testing performance
- technological improvements

Important:
The presence of a company, manufacturer, or technology organisation does NOT automatically indicate Business.

Choose Technology when the main question is:

"How does the technology work, develop, or improve?"


--------------------------------------------------


Safety and Risk

Choose this topic when the article mainly focuses on:

- crashes
- failures
- safety concerns
- reliability problems
- operational risks
- safety evaluation

Important:
Safety remains the primary topic even if companies or public reactions are mentioned.

Do not choose Public Acceptance unless the article mainly discusses human attitudes or trust.


--------------------------------------------------


Policy and Regulation

Choose this topic when the article mainly focuses on:

- government policies
- legislation
- standards
- regulatory approval
- permits
- official rules or decisions


--------------------------------------------------


Business and Commercialisation

Choose this topic when the article mainly focuses on:

- company strategy
- investment
- acquisitions
- partnerships
- competition
- financial performance
- business models
- industrial or market development

Important:
Do NOT classify as Business only because:

- a company is mentioned;
- an AV test occurs;
- a service is launched;
- a company operates autonomous vehicles.

Choose Business only when the central issue is:

"How companies, markets, or industries develop AV-related business."


--------------------------------------------------


Public Acceptance and Trust

Choose this topic when the article mainly focuses on:

- public opinion
- consumer attitudes
- trust
- concerns about adoption
- willingness to use autonomous vehicles

Important:
The presence of words such as "concern", "fear", or "skepticism" does not automatically indicate Public Acceptance.

Choose Public Acceptance only when human attitudes or adoption behaviour are the main focus.


--------------------------------------------------


Mobility and Social Impact

Choose this topic when the article mainly focuses on:

- robotaxi services
- autonomous ride-hailing
- passenger transportation
- mobility solutions
- changes in transportation systems
- accessibility
- effects on daily life or society

Important:
Robotaxi and autonomous transportation services should usually be classified as Mobility and Social Impact.

Only choose Business when the article mainly discusses corporate strategy, investment, or market competition.


--------------------------------------------------


Environment and Sustainability

Choose this topic when the article mainly focuses on:

- emissions
- energy efficiency
- environmental benefits
- sustainability


--------------------------------------------------


Legal and Ethics

Choose this topic when the article mainly focuses on:

- liability
- responsibility
- legal disputes
- ethical dilemmas
- accountability


--------------------------------------------------


Primary topic selection rules:


1.
Choose the topic representing the central framing of the article.

2.
Do not classify based on isolated keywords.

3.
When multiple themes appear, select the issue receiving the greatest emphasis.

4.
Distinguish carefully between:

- technical development → Technology and Innovation

- company strategy or market activity → Business and Commercialisation

- passenger services and transportation changes → Mobility and Social Impact

- government rules and approvals → Policy and Regulation

- accidents and failures → Safety and Risk


5.
Objective reporting does not automatically mean Business or Technology.
Classify according to the substantive issue being discussed.

6.
If autonomous vehicles are only mentioned briefly without a clear thematic focus, choose Other.


Return only ONE topic name.


Text:

{text}

"""

In [74]:
# %%
# ============================================================
# Test P3-R
# ============================================================


for idx,row in validation.head(5).iterrows():

    print("="*80)

    print("Source:", row["source"])

    print("Human:",
          row["human_topic"])

    prediction = classify_topic(
        build_prompt_p3r(row["text"])
    )

    print("P3-R:",
          prediction)

Source: USA Today
Human: Technology and Innovation
P3-R: Business and Commercialisation
Source: USA Today
Human: Business and Commercialisation
P3-R: Business and Commercialisation
Source: USA Today
Human: Safety and Risk
P3-R: Public Acceptance and Trust
Source: USA Today
Human: Mobility and Social Impact
P3-R: Public Acceptance and Trust
Source: USA Today
Human: Technology and Innovation
P3-R: Business and Commercialisation


In [75]:
# %%
# ============================================================
# Run P3-R on full validation sample
# ============================================================


results["P3R_EN"] = None


for idx,row in results.iterrows():

    print(
        f"Processing P3-R {idx+1}/{len(results)}"
    )


    results.at[idx,"P3R_EN"] = classify_topic(
        build_prompt_p3r(
            row["text"]
        )
    )


print("P3-R completed.")

Processing P3-R 1/66
Processing P3-R 2/66
Processing P3-R 3/66
Processing P3-R 4/66
Processing P3-R 5/66
Processing P3-R 6/66
Processing P3-R 7/66
Processing P3-R 8/66
Processing P3-R 9/66
Processing P3-R 10/66
Processing P3-R 11/66
Processing P3-R 12/66
Processing P3-R 13/66
Processing P3-R 14/66
Processing P3-R 15/66
Processing P3-R 16/66
Processing P3-R 17/66
Processing P3-R 18/66
Processing P3-R 19/66
Processing P3-R 20/66
Processing P3-R 21/66
Processing P3-R 22/66
Processing P3-R 23/66
Processing P3-R 24/66
Processing P3-R 25/66
Processing P3-R 26/66
Processing P3-R 27/66
Processing P3-R 28/66
Processing P3-R 29/66
Processing P3-R 30/66
Processing P3-R 31/66
Processing P3-R 32/66
Processing P3-R 33/66
Processing P3-R 34/66
Processing P3-R 35/66
Processing P3-R 36/66
Processing P3-R 37/66
Processing P3-R 38/66
Processing P3-R 39/66
Processing P3-R 40/66
Processing P3-R 41/66
Processing P3-R 42/66
Processing P3-R 43/66
Processing P3-R 44/66
Processing P3-R 45/66
Processing P3-R 46/

In [76]:
# %%
# ============================================================
# Validate model output labels
#
# Check whether GPT returned anything outside the predefined
# topic categories.
# ============================================================


for prompt in [

    "P1_EN",

    "P2_EN",

    "P3_EN",

    "P3R_EN"

]:


    if prompt not in results.columns:

        continue


    unexpected = (

        results.loc[
            results[prompt].notna()
            &
            ~results[prompt].isin(TOPICS),
            prompt
        ]
        .value_counts()

    )


    print(
        "\n" + "=" * 60
    )

    print(
        "Checking:",
        prompt
    )


    if len(unexpected) == 0:

        print(
            "All predictions are valid topic labels."
        )

    else:

        print(
            "WARNING: Unexpected model outputs found:"
        )

        display(
            unexpected
        )


Checking: P1_EN
All predictions are valid topic labels.

Checking: P2_EN
All predictions are valid topic labels.

Checking: P3_EN
All predictions are valid topic labels.

Checking: P3R_EN
All predictions are valid topic labels.


In [77]:
# %%
# ============================================================
# Compare P3 and P3-R
#
# Use exactly the same evaluation criteria as the original
# P1 / P2 / P3 comparison.
# ============================================================


from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    cohen_kappa_score

)


prompt_columns = [

    "P3_EN",

    "P3R_EN"

]


comparison_rows = []



for prompt in prompt_columns:


    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt
    ]


    comparison_rows.append({

        "prompt":
            prompt,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df = comparison_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


display(
    comparison_df.round(4)
)

,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P3_EN,66,0.7879,0.669,0.6981,0.6543,0.7316
1,P3R_EN,66,0.7424,0.658,0.6758,0.6536,0.6733


In [78]:
# %%
# ============================================================
# Confusion matrix P3-R
# ============================================================


from sklearn.metrics import confusion_matrix


cm_p3r = confusion_matrix(

    results["human_topic"],

    results["P3R_EN"],

    labels=TOPICS

)



cm_p3r_df = pd.DataFrame(

    cm_p3r,

    index=TOPICS,

    columns=TOPICS

)



display(
    cm_p3r_df
)

,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,9,1,0,3,0,1,0,0,0
Safety and Risk,0,6,0,2,0,0,0,0,0
Policy and Regulation,0,0,3,1,0,0,0,0,0
Business and Commercialisation,0,0,0,19,0,3,0,0,0
Public Acceptance and Trust,0,0,0,1,1,0,0,0,1
Mobility and Social Impact,0,0,0,0,1,8,0,0,0
Environment and Sustainability,0,0,0,0,0,0,0,0,0
Legal and Ethics,0,0,0,0,0,0,0,3,0
Other,1,0,0,0,1,0,0,0,0


In [79]:
# %%
# ============================================================
# P3-R errors
# ============================================================


p3r_errors = results[
    results["human_topic"]
    !=
    results["P3R_EN"]
].copy()



print(
    "Number of P3-R errors:",
    len(p3r_errors)
)



display(

    p3r_errors[
        [
            "source",
            "text",
            "human_topic",
            "P3_EN",
            "P3R_EN"
        ]
    ]

)

Number of P3-R errors: 17


,source,text,human_topic,P3_EN,P3R_EN
3,USA Today,So if you're skeptical or nervous about the co...,Mobility and Social Impact,Public Acceptance and Trust,Public Acceptance and Trust
4,USA Today,General Motors said Tuesday it has finished ma...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
9,USA Today,That's because self-driving cars are safer and...,Technology and Innovation,Safety and Risk,Safety and Risk
14,USA Today,Musk has also lashed out against the media in ...,Other,Public Acceptance and Trust,Public Acceptance and Trust
19,USA Today,"Yes, Musk has repeatedly promised that Teslas ...",Business and Commercialisation,Mobility and Social Impact,Mobility and Social Impact
21,USA Today,"Granted, it can be tougher to assess how impre...",Unclear,Technology and Innovation,Technology and Innovation
23,USA Today,Uber will roll out two pilot programs in Los A...,Business and Commercialisation,Business and Commercialisation,Mobility and Social Impact
27,USA Today,Stellantis has shelved its first Level 3 advan...,Technology and Innovation,Business and Commercialisation,Business and Commercialisation
29,WSJ,Highway planners are preparing for future wher...,Technology and Innovation,Technology and Innovation,Mobility and Social Impact
31,WSJ,Uber Technologies has resumed test using its d...,Safety and Risk,Business and Commercialisation,Business and Commercialisation


In [80]:
# %%
# ============================================================
# Cases improved / worsened by P3-R
# ============================================================


comparison = results.copy()


comparison["P3_correct"] = (

    comparison["P3_EN"]
    ==
    comparison["human_topic"]

)


comparison["P3R_correct"] = (

    comparison["P3R_EN"]
    ==
    comparison["human_topic"]

)



fixed = comparison[
    (~comparison["P3_correct"])
    &
    (comparison["P3R_correct"])
]


worsened = comparison[
    (comparison["P3_correct"])
    &
    (~comparison["P3R_correct"])
]



print(
    "Fixed by P3-R:",
    len(fixed)
)


print(
    "Worsened by P3-R:",
    len(worsened)
)


display(
    fixed[
        [
            "text",
            "human_topic",
            "P3_EN",
            "P3R_EN"
        ]
    ]
)


display(
    worsened[
        [
            "text",
            "human_topic",
            "P3_EN",
            "P3R_EN"
        ]
    ]
)

Fixed by P3-R: 2
Worsened by P3-R: 5


,text,human_topic,P3_EN,P3R_EN
5,"Benchmark, one of the company's largest invest...",Legal and Ethics,Business and Commercialisation,Legal and Ethics
16,As self-driving vehicle experiments are launch...,Business and Commercialisation,Technology and Innovation,Business and Commercialisation


,text,human_topic,P3_EN,P3R_EN
4,General Motors said Tuesday it has finished ma...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
23,Uber will roll out two pilot programs in Los A...,Business and Commercialisation,Business and Commercialisation,Mobility and Social Impact
29,Highway planners are preparing for future wher...,Technology and Innovation,Technology and Innovation,Mobility and Social Impact
45,Ford Motor will team up with Chinese search en...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
61,HAMBURG—The self-driving taxi carefully steere...,Safety and Risk,Safety and Risk,Business and Commercialisation


In [81]:
# %%
# ============================================================
# Save P3-R validation results
# ============================================================


OUTPUT = (

BASE_DIR /

"english_primary_topic_prompt_validation_with_P3R.xlsx"

)



results.to_excel(

    OUTPUT,

    index=False

)



print(
    OUTPUT
)

/Users/yurujia/Desktop/Dissertation Data/USA/english_primary_topic_prompt_validation_with_P3R.xlsx


In [82]:
# %%
# ============================================================
# Per-topic classification performance
#
# Purpose:
# Examine Precision / Recall / F1 / Support for each topic
# for P3_EN and P3R_EN
# ============================================================

from sklearn.metrics import classification_report


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def get_per_topic_metrics(df, prompt_col, topic_labels):

    valid = df[
        df[prompt_col].notna()
        &
        df["human_topic"].notna()
    ].copy()

    y_true = valid["human_topic"]
    y_pred = valid[prompt_col]

    report = classification_report(
        y_true,
        y_pred,
        labels=topic_labels,
        target_names=topic_labels,
        output_dict=True,
        zero_division=0
    )

    # Keep only actual topic categories
    rows = []

    for topic in topic_labels:

        metrics = report[topic]

        rows.append({
            "prompt": prompt_col,
            "topic": topic,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1-score"],
            "support": int(metrics["support"])
        })

    return pd.DataFrame(rows)


# %%
# ============================================================
# Per-topic metrics for P3_EN
# ============================================================

p3_per_topic = get_per_topic_metrics(
    results,
    "P3_EN",
    TOPICS
)

display(
    p3_per_topic.round(4)
)


# %%
# ============================================================
# Per-topic metrics for P3R_EN
# ============================================================

p3r_per_topic = get_per_topic_metrics(
    results,
    "P3R_EN",
    TOPICS
)

display(
    p3r_per_topic.round(4)
)


# %%
# ============================================================
# Combine P3 and P3-R
# ============================================================

per_topic_comparison = pd.concat(
    [
        p3_per_topic,
        p3r_per_topic
    ],
    ignore_index=True
)

display(
    per_topic_comparison.round(4)
)


# %%
# ============================================================
# Human topic support
#
# Important because low-support topics can have unstable F1
# ============================================================

topic_support = (
    results["human_topic"]
    .value_counts()
    .reindex(TOPICS, fill_value=0)
    .reset_index()
)

topic_support.columns = [
    "topic",
    "human_support"
]

display(topic_support)


# %%
# ============================================================
# Normalised confusion matrix for P3_EN
#
# Each row sums to 1.
# Shows where each human topic is being misclassified.
# ============================================================

from sklearn.metrics import confusion_matrix


cm_p3_normalised = confusion_matrix(
    results["human_topic"],
    results["P3_EN"],
    labels=TOPICS,
    normalize="true"
)

cm_p3_normalised_df = pd.DataFrame(
    cm_p3_normalised,
    index=TOPICS,
    columns=TOPICS
)

display(
    cm_p3_normalised_df.round(3)
)


# %%
# ============================================================
# Normalised confusion matrix for P3R_EN
# ============================================================

cm_p3r_normalised = confusion_matrix(
    results["human_topic"],
    results["P3R_EN"],
    labels=TOPICS,
    normalize="true"
)

cm_p3r_normalised_df = pd.DataFrame(
    cm_p3r_normalised,
    index=TOPICS,
    columns=TOPICS
)

display(
    cm_p3r_normalised_df.round(3)
)


# %%
# ============================================================
# Identify weak-performing topics
#
# Threshold is only for diagnostic purposes.
# Do NOT treat 0.50 as a universal statistical cutoff.
# ============================================================

weak_topics_p3 = p3_per_topic[
    p3_per_topic["f1_score"] < 0.50
].copy()

print("P3_EN topics with F1 < 0.50:")

display(
    weak_topics_p3.round(4)
)


# %%
# ============================================================
# Compare per-topic F1 change from P3 to P3-R
# ============================================================

f1_change = (
    p3_per_topic[
        ["topic", "f1_score"]
    ]
    .rename(
        columns={
            "f1_score": "P3_F1"
        }
    )
    .merge(
        p3r_per_topic[
            ["topic", "f1_score"]
        ].rename(
            columns={
                "f1_score": "P3R_F1"
            }
        ),
        on="topic",
        how="outer"
    )
)

f1_change["F1_change_P3R_minus_P3"] = (
    f1_change["P3R_F1"]
    -
    f1_change["P3_F1"]
)

display(
    f1_change.round(4)
)


# %%
# ============================================================
# Save detailed diagnostic results
# ============================================================

DIAGNOSTIC_OUTPUT = (
    BASE_DIR /
    "english_topic_prompt_detailed_diagnostics.xlsx"
)


with pd.ExcelWriter(
    DIAGNOSTIC_OUTPUT,
    engine="openpyxl"
) as writer:

    comparison_df.to_excel(
        writer,
        sheet_name="overall_metrics",
        index=False
    )

    p3_per_topic.to_excel(
        writer,
        sheet_name="P3_per_topic",
        index=False
    )

    p3r_per_topic.to_excel(
        writer,
        sheet_name="P3R_per_topic",
        index=False
    )

    f1_change.to_excel(
        writer,
        sheet_name="F1_comparison",
        index=False
    )

    topic_support.to_excel(
        writer,
        sheet_name="topic_support",
        index=False
    )

    cm_df.to_excel(
        writer,
        sheet_name="P3_confusion_raw"
    )

    cm_p3_normalised_df.to_excel(
        writer,
        sheet_name="P3_confusion_normalised"
    )

    cm_p3r_df.to_excel(
        writer,
        sheet_name="P3R_confusion_raw"
    )

    cm_p3r_normalised_df.to_excel(
        writer,
        sheet_name="P3R_confusion_normalised"
    )

    weak_topics_p3.to_excel(
        writer,
        sheet_name="P3_weak_topics",
        index=False
    )


print(
    "Detailed diagnostics saved to:"
)

print(
    DIAGNOSTIC_OUTPUT
)

,prompt,topic,precision,recall,f1_score,support
0,P3_EN,Technology and Innovation,0.7500,0.8571,0.8000,14
1,P3_EN,Safety and Risk,0.8750,0.8750,0.8750,8
2,P3_EN,Policy and Regulation,1.0000,0.7500,0.8571,4
3,P3_EN,Business and Commercialisation,0.8261,0.8636,0.8444,22
4,P3_EN,Public Acceptance and Trust,0.3333,0.3333,0.3333,3
5,P3_EN,Mobility and Social Impact,0.8000,0.8889,0.8421,9
6,P3_EN,Environment and Sustainability,0.0000,0.0000,0.0000,0
7,P3_EN,Legal and Ethics,1.0000,0.6667,0.8000,3
8,P3_EN,Other,0.0000,0.0000,0.0000,2


,prompt,topic,precision,recall,f1_score,support
0,P3R_EN,Technology and Innovation,0.8182,0.6429,0.7200,14
1,P3R_EN,Safety and Risk,0.8571,0.7500,0.8000,8
2,P3R_EN,Policy and Regulation,1.0000,0.7500,0.8571,4
3,P3R_EN,Business and Commercialisation,0.7308,0.8636,0.7917,22
4,P3R_EN,Public Acceptance and Trust,0.3333,0.3333,0.3333,3
5,P3R_EN,Mobility and Social Impact,0.6667,0.8889,0.7619,9
6,P3R_EN,Environment and Sustainability,0.0000,0.0000,0.0000,0
7,P3R_EN,Legal and Ethics,1.0000,1.0000,1.0000,3
8,P3R_EN,Other,0.0000,0.0000,0.0000,2


,prompt,topic,precision,recall,f1_score,support
0,P3_EN,Technology and Innovation,0.7500,0.8571,0.8000,14
1,P3_EN,Safety and Risk,0.8750,0.8750,0.8750,8
2,P3_EN,Policy and Regulation,1.0000,0.7500,0.8571,4
3,P3_EN,Business and Commercialisation,0.8261,0.8636,0.8444,22
4,P3_EN,Public Acceptance and Trust,0.3333,0.3333,0.3333,3
5,P3_EN,Mobility and Social Impact,0.8000,0.8889,0.8421,9
6,P3_EN,Environment and Sustainability,0.0000,0.0000,0.0000,0
7,P3_EN,Legal and Ethics,1.0000,0.6667,0.8000,3
8,P3_EN,Other,0.0000,0.0000,0.0000,2
9,P3R_EN,Technology and Innovation,0.8182,0.6429,0.7200,14


,topic,human_support
0,Technology and Innovation,14
1,Safety and Risk,8
2,Policy and Regulation,4
3,Business and Commercialisation,22
4,Public Acceptance and Trust,3
5,Mobility and Social Impact,9
6,Environment and Sustainability,0
7,Legal and Ethics,3
8,Other,2


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.857,0.071,0.00,0.071,0.000,0.000,0.0,0.000,0.000
Safety and Risk,0.000,0.875,0.00,0.125,0.000,0.000,0.0,0.000,0.000
Policy and Regulation,0.250,0.000,0.75,0.000,0.000,0.000,0.0,0.000,0.000
Business and Commercialisation,0.045,0.000,0.00,0.864,0.000,0.091,0.0,0.000,0.000
Public Acceptance and Trust,0.000,0.000,0.00,0.333,0.333,0.000,0.0,0.000,0.333
Mobility and Social Impact,0.000,0.000,0.00,0.000,0.111,0.889,0.0,0.000,0.000
Environment and Sustainability,0.000,0.000,0.00,0.000,0.000,0.000,0.0,0.000,0.000
Legal and Ethics,0.000,0.000,0.00,0.333,0.000,0.000,0.0,0.667,0.000
Other,0.500,0.000,0.00,0.000,0.500,0.000,0.0,0.000,0.000


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.643,0.071,0.00,0.214,0.000,0.071,0.0,0.0,0.000
Safety and Risk,0.000,0.750,0.00,0.250,0.000,0.000,0.0,0.0,0.000
Policy and Regulation,0.000,0.000,0.75,0.250,0.000,0.000,0.0,0.0,0.000
Business and Commercialisation,0.000,0.000,0.00,0.864,0.000,0.136,0.0,0.0,0.000
Public Acceptance and Trust,0.000,0.000,0.00,0.333,0.333,0.000,0.0,0.0,0.333
Mobility and Social Impact,0.000,0.000,0.00,0.000,0.111,0.889,0.0,0.0,0.000
Environment and Sustainability,0.000,0.000,0.00,0.000,0.000,0.000,0.0,0.0,0.000
Legal and Ethics,0.000,0.000,0.00,0.000,0.000,0.000,0.0,1.0,0.000
Other,0.500,0.000,0.00,0.000,0.500,0.000,0.0,0.0,0.000


P3_EN topics with F1 < 0.50:


,prompt,topic,precision,recall,f1_score,support
4,P3_EN,Public Acceptance and Trust,0.3333,0.3333,0.3333,3
6,P3_EN,Environment and Sustainability,0.0000,0.0000,0.0000,0
8,P3_EN,Other,0.0000,0.0000,0.0000,2


,topic,P3_F1,P3R_F1,F1_change_P3R_minus_P3
0,Business and Commercialisation,0.8444,0.7917,-0.0528
1,Environment and Sustainability,0.0000,0.0000,0.0000
2,Legal and Ethics,0.8000,1.0000,0.2000
3,Mobility and Social Impact,0.8421,0.7619,-0.0802
4,Other,0.0000,0.0000,0.0000
5,Policy and Regulation,0.8571,0.8571,0.0000
6,Public Acceptance and Trust,0.3333,0.3333,0.0000
7,Safety and Risk,0.8750,0.8000,-0.0750
8,Technology and Innovation,0.8000,0.7200,-0.0800


Detailed diagnostics saved to:
/Users/yurujia/Desktop/Dissertation Data/USA/english_topic_prompt_detailed_diagnostics.xlsx


In [83]:
# %%
# ============================================================
# Supplementary Macro F1 diagnostics
#
# 1. Observed-category Macro F1:
#    Includes all categories represented in human validation,
#    including "Other".
#
# 2. Substantive-topic Macro F1:
#    Excludes the residual "Other" category.
#
# The second measure is supplementary only.
# Main results should still report the observed-category
# Macro F1 used in the main evaluation table.
# ============================================================


substantive_observed_topics = [

    topic

    for topic in observed_topics

    if topic != "Other"

]


supplementary_rows = []



for prompt in [

    "P3_EN",

    "P3R_EN"

]:


    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt
    ]


    observed_macro_f1 = f1_score(

        y_true,

        y_pred,

        labels=observed_topics,

        average="macro",

        zero_division=0

    )


    substantive_macro_f1 = f1_score(

        y_true,

        y_pred,

        labels=substantive_observed_topics,

        average="macro",

        zero_division=0

    )


    supplementary_rows.append({

        "prompt":
            prompt,

        "observed_category_macro_f1":
            observed_macro_f1,

        "substantive_topic_macro_f1_excluding_other":
            substantive_macro_f1,

        "number_observed_categories":
            len(observed_topics),

        "number_substantive_categories":
            len(substantive_observed_topics)

    })


supplementary_f1_df = pd.DataFrame(
    supplementary_rows
)


display(
    supplementary_f1_df.round(4)
)

,prompt,observed_category_macro_f1,substantive_topic_macro_f1_excluding_other,number_observed_categories,number_substantive_categories
0,P3_EN,0.669,0.7646,8,7
1,P3R_EN,0.658,0.7520,8,7
